## Creation of the Folder

In [1]:
import os
os.makedirs("data", exist_ok=True)
print("Folder created at:", os.path.abspath("data"))

Folder created at: C:\Users\ADMIN\data


## Customers table 

In [2]:
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

# Use a relative path so it works on your machine
os.makedirs("data", exist_ok=True)
OUT_DIR = "data"

RNG = np.random.default_rng(42)
N_CUSTOMERS = 10000

customer_id = np.arange(100000, 100000 + N_CUSTOMERS)

channel = RNG.choice(["digital", "sales_assisted"], size=N_CUSTOMERS, p=[0.62, 0.38])

regions = ["Nairobi", "Nakuru", "Mombasa", "Kisumu", "Eldoret", "Kakamega",
           "Meru", "Machakos", "Kiambu", "West Pokot", "Uasin Gishu", "Nyeri"]
region = RNG.choice(regions, size=N_CUSTOMERS,
                     p=[0.28, 0.10, 0.12, 0.08, 0.06, 0.05, 0.05, 0.05, 0.09, 0.03, 0.05, 0.04])

employment_type = RNG.choice(
    ["formal_employed", "self_employed_business", "informal_gig", "civil_servant"],
    size=N_CUSTOMERS, p=[0.30, 0.32, 0.28, 0.10]
)

age = np.clip(RNG.normal(33, 9, N_CUSTOMERS), 18, 65).astype(int)
gender = RNG.choice(["M", "F"], size=N_CUSTOMERS, p=[0.54, 0.46])

income_base = {
    "formal_employed": 45000, "self_employed_business": 38000,
    "informal_gig": 22000, "civil_servant": 40000
}
monthly_income = np.array([
    max(8000, RNG.normal(income_base[e], income_base[e] * 0.4))
    for e in employment_type
]).round(-2)

signup_date = pd.to_datetime("2023-01-01") + pd.to_timedelta(
    RNG.integers(0, 900, N_CUSTOMERS), unit="D"
)

customers = pd.DataFrame({
    "customer_id": customer_id,
    "acquisition_channel": channel,
    "region": region,
    "employment_type": employment_type,
    "age": age,
    "gender": gender,
    "monthly_income_kes": monthly_income,
    "signup_date": signup_date,
})

customers.to_csv(f"{OUT_DIR}/customers.csv", index=False)
print(customers.shape)
customers.head()

(10000, 8)


,customer_id,acquisition_channel,region,employment_type,age,gender,monthly_income_kes,signup_date
0,100000,sales_assisted,Meru,informal_gig,18,M,24500.0,2025-01-11
1,100001,digital,Meru,civil_servant,25,F,30300.0,2025-04-25
2,100002,sales_assisted,Nairobi,formal_employed,24,M,31300.0,2024-10-01
3,100003,sales_assisted,Nairobi,formal_employed,26,F,75700.0,2023-08-23
4,100004,digital,Nakuru,self_employed_business,35,F,37500.0,2023-01-04


## CRB DATA (Credit Reference Bureau snapshot)

In [3]:
base_crb = RNG.normal(600, 110, N_CUSTOMERS)
base_crb += (monthly_income - 30000) / 1000
base_crb += np.where(employment_type == "informal_gig", -25, 0)
base_crb += np.where(employment_type == "civil_servant", 20, 0)
crb_score = np.clip(base_crb, 200, 900).round().astype(int)

num_active_loans_other_lenders = RNG.poisson(
    lam=np.clip((900 - crb_score) / 220, 0.1, None)
)
total_outstanding_debt_kes = (num_active_loans_other_lenders *
                               RNG.uniform(3000, 25000, N_CUSTOMERS)).round(-2)

default_propensity = np.clip((650 - crb_score) / 500, 0, None)
num_defaults_last_24m = RNG.poisson(lam=default_propensity * 1.3)
max_days_past_due_24m = np.where(
    num_defaults_last_24m > 0,
    RNG.integers(30, 180, N_CUSTOMERS),
    RNG.integers(0, 15, N_CUSTOMERS)
)
crb_listed_negative = (num_defaults_last_24m >= 1).astype(int)

crb_data = pd.DataFrame({
    "customer_id": customer_id,
    "crb_score": crb_score,
    "num_active_loans_other_lenders": num_active_loans_other_lenders,
    "total_outstanding_debt_kes": total_outstanding_debt_kes,
    "num_defaults_last_24m": num_defaults_last_24m,
    "max_days_past_due_24m": max_days_past_due_24m,
    "crb_listed_negative": crb_listed_negative,
    "crb_pulled_date": signup_date + pd.to_timedelta(RNG.integers(0, 5, N_CUSTOMERS), unit="D"),
})

crb_data.to_csv(f"{OUT_DIR}/crb_data.csv", index=False)
print(crb_data.shape)
crb_data.head()

(10000, 8)


,customer_id,crb_score,num_active_loans_other_lenders,total_outstanding_debt_kes,num_defaults_last_24m,max_days_past_due_24m,crb_listed_negative,crb_pulled_date
0,100000,597,3,28800.0,2,38,1,2025-01-11
1,100001,434,1,24500.0,1,120,1,2025-04-27
2,100002,508,0,0.0,1,92,1,2024-10-02
3,100003,497,3,29700.0,0,6,0,2023-08-27
4,100004,662,0,0.0,0,4,0,2023-01-07


## BANK STATEMENT FEATURES (aggregated, as a scoring model would consume)

In [4]:
salary_regularity_score = np.clip(
    RNG.normal(
        np.where(np.isin(employment_type, ["formal_employed", "civil_servant"]), 0.85, 0.55),
        0.15
    ), 0.05, 1.0
)

avg_monthly_inflow_kes = np.clip(
    monthly_income * RNG.normal(1.15, 0.3, N_CUSTOMERS), 5000, None
).round(-2)
avg_monthly_outflow_kes = np.clip(
    avg_monthly_inflow_kes * RNG.normal(0.88, 0.15, N_CUSTOMERS), 3000, None
).round(-2)

inflow_volatility_cv = np.clip(RNG.normal(0.35, 0.15, N_CUSTOMERS) *
                                (1.4 - salary_regularity_score), 0.03, 1.5).round(3)

avg_closing_balance_kes = np.clip(
    (avg_monthly_inflow_kes - avg_monthly_outflow_kes) * RNG.uniform(0.5, 3, N_CUSTOMERS) + 1500,
    -5000, None
).round(-2)

num_bounced_payments_6m = RNG.poisson(
    lam=np.clip(1.2 - salary_regularity_score, 0.02, None) * 2.5
)

mobile_money_txn_count_monthly = RNG.poisson(lam=RNG.uniform(15, 60, N_CUSTOMERS))
num_distinct_income_sources = RNG.choice([1, 2, 3], size=N_CUSTOMERS, p=[0.68, 0.25, 0.07])

bank_statement_features = pd.DataFrame({
    "customer_id": customer_id,
    "avg_monthly_inflow_kes": avg_monthly_inflow_kes,
    "avg_monthly_outflow_kes": avg_monthly_outflow_kes,
    "inflow_volatility_cv": inflow_volatility_cv,
    "avg_closing_balance_kes": avg_closing_balance_kes,
    "salary_regularity_score": salary_regularity_score.round(3),
    "num_bounced_payments_6m": num_bounced_payments_6m,
    "mobile_money_txn_count_monthly": mobile_money_txn_count_monthly,
    "num_distinct_income_sources": num_distinct_income_sources,
    "statement_period_months": RNG.choice([3, 6, 12], size=N_CUSTOMERS, p=[0.2, 0.5, 0.3]),
})

bank_statement_features.to_csv(f"{OUT_DIR}/bank_statement_features.csv", index=False)
print(bank_statement_features.shape)
bank_statement_features.head()

(10000, 10)


,customer_id,avg_monthly_inflow_kes,avg_monthly_outflow_kes,inflow_volatility_cv,avg_closing_balance_kes,salary_regularity_score,num_bounced_payments_6m,mobile_money_txn_count_monthly,num_distinct_income_sources,statement_period_months
0,100000,28300.0,25600.0,0.514,9600.0,0.356,5,35,3,6
1,100001,41700.0,33100.0,0.185,19400.0,1.000,0,31,1,6
2,100002,36600.0,31000.0,0.177,13700.0,1.000,0,46,2,6
3,100003,84200.0,79700.0,0.226,14700.0,0.979,0,48,1,12
4,100004,30100.0,30500.0,0.254,1100.0,0.490,2,36,1,12


## APP BEHAVIOURAL SIGNALS (rich for digital, sparse/missing for sales_assisted)

In [5]:

is_digital = (channel == "digital")

app_tenure_days = np.where(
    is_digital, np.clip(RNG.normal(220, 160, N_CUSTOMERS), 1, 1000).astype(int), np.nan
)
app_sessions_per_week = np.where(
    is_digital, np.clip(RNG.poisson(lam=5, size=N_CUSTOMERS), 0, None), np.nan
)
device_type = np.where(
    is_digital, RNG.choice(["android_low_end", "android_mid", "android_high", "ios"],
                            size=N_CUSTOMERS, p=[0.35, 0.40, 0.15, 0.10]), None
)
contacts_permission_granted = np.where(
    is_digital, RNG.choice([1, 0], size=N_CUSTOMERS, p=[0.55, 0.45]), np.nan
)
sms_permission_granted = np.where(
    is_digital, RNG.choice([1, 0], size=N_CUSTOMERS, p=[0.6, 0.4]), np.nan
)
num_support_tickets_90d = np.where(
    is_digital, RNG.poisson(lam=0.4, size=N_CUSTOMERS), np.nan
)
profile_completeness_pct = np.where(
    is_digital, np.clip(RNG.normal(78, 18, N_CUSTOMERS), 20, 100).round(1), np.nan
)
days_since_last_app_open = np.where(
    is_digital, RNG.poisson(lam=4, size=N_CUSTOMERS), np.nan
)

app_behavior = pd.DataFrame({
    "customer_id": customer_id,
    "app_tenure_days": app_tenure_days,
    "app_sessions_per_week": app_sessions_per_week,
    "device_type": device_type,
    "contacts_permission_granted": contacts_permission_granted,
    "sms_permission_granted": sms_permission_granted,
    "num_support_tickets_90d": num_support_tickets_90d,
    "profile_completeness_pct": profile_completeness_pct,
    "days_since_last_app_open": days_since_last_app_open,
})

app_behavior.to_csv(f"{OUT_DIR}/app_behavior.csv", index=False)
print(app_behavior.shape)
print("\nMissing values (should roughly match the sales_assisted share, ~38%):")
print(app_behavior.isna().mean().round(3))
app_behavior.head(8)

(10000, 9)

Missing values (should roughly match the sales_assisted share, ~38%):
customer_id                    0.000
app_tenure_days                0.375
app_sessions_per_week          0.375
device_type                    0.375
contacts_permission_granted    0.375
sms_permission_granted         0.375
num_support_tickets_90d        0.375
profile_completeness_pct       0.375
days_since_last_app_open       0.375
dtype: float64


,customer_id,app_tenure_days,app_sessions_per_week,device_type,contacts_permission_granted,sms_permission_granted,num_support_tickets_90d,profile_completeness_pct,days_since_last_app_open
0,100000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100001,419.0,6.0,android_mid,0.0,1.0,1.0,43.8,3.0
2,100002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100004,314.0,5.0,android_high,1.0,0.0,0.0,97.6,4.0
5,100005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,100006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,100007,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## LOAN APPLICATIONS (each customer can have 1-4 applications over time)

In [6]:
records = []
loan_id_counter = 500000

for i in range(N_CUSTOMERS):
    cid = customer_id[i]
    n_apps = RNG.choice([1, 2, 3, 4], p=[0.42, 0.31, 0.18, 0.09])
    app_date = customers.loc[i, "signup_date"] + timedelta(days=int(RNG.integers(0, 10)))

    prior_completed = 0
    prior_defaults = 0
    prior_avg_days_late = 0.0
    prior_repayment_rate = 1.0

    for k in range(n_apps):
        if k > 0:
            app_date = app_date + timedelta(days=int(RNG.integers(25, 120)))
        if app_date > pd.Timestamp("2026-06-15"):
            break

        loan_id_counter += 1

        requested_amount = float(np.clip(
            RNG.normal(monthly_income[i] * RNG.uniform(0.3, 1.2), 3000), 2000, 150000
        ).round(-2))

        z = 0.0
        z += (650 - crb_score[i]) / 90.0
        z += num_defaults_last_24m[i] * 0.55
        z += (max_days_past_due_24m[i] / 100.0) * 0.4
        z += (1 - salary_regularity_score[i]) * 1.4
        z += inflow_volatility_cv[i] * 1.1
        z += num_bounced_payments_6m[i] * 0.22
        z += -0.35 if employment_type[i] in ("formal_employed", "civil_servant") else 0.25
        z += prior_defaults * 0.9
        z += (1 - prior_repayment_rate) * 1.6
        z += prior_avg_days_late / 60.0
        z += 0.15 if channel[i] == "sales_assisted" else -0.05
        z += RNG.normal(0, 0.65)

        default_prob = 1 / (1 + np.exp(-(z - 1.1)))
        default_prob = float(np.clip(default_prob, 0.01, 0.9))

        if default_prob < 0.12:
            risk_tier = "A"
        elif default_prob < 0.25:
            risk_tier = "B"
        elif default_prob < 0.45:
            risk_tier = "C"
        else:
            risk_tier = "D"

        approve_prob = {"A": 0.97, "B": 0.90, "C": 0.68, "D": 0.30}[risk_tier]
        approved = RNG.random() < approve_prob

        if approved:
            approval_ratio = {"A": 1.0, "B": 0.9, "C": 0.7, "D": 0.5}[risk_tier]
            approved_amount = round(requested_amount * approval_ratio * RNG.uniform(0.9, 1.0), -2)
            interest_rate_monthly_pct = {
                "A": RNG.uniform(6, 9), "B": RNG.uniform(9, 13),
                "C": RNG.uniform(13, 18), "D": RNG.uniform(18, 24)
            }[risk_tier]
            tenor_days = int(RNG.choice([30, 60, 90, 120]))

            disbursed_date = app_date + timedelta(days=int(RNG.integers(0, 2)))
            matured = (pd.Timestamp("2026-07-03") - disbursed_date).days > tenor_days + 5

            if matured:
                defaulted = RNG.random() < default_prob
                if defaulted:
                    loan_status = "written_off" if RNG.random() < 0.35 else "closed_default"
                    days_late = int(RNG.integers(31, 200))
                    repaid_pct = round(float(RNG.uniform(0, 0.6)), 3)
                else:
                    loan_status = "closed_good"
                    days_late = int(max(0, RNG.normal(2, 6)))
                    repaid_pct = 1.0
            else:
                loan_status = "active"
                defaulted = None
                days_late = int(max(0, RNG.normal(1, 4)))
                repaid_pct = round(float(RNG.uniform(0, 0.9)), 3)
        else:
            approved_amount = 0.0
            interest_rate_monthly_pct = np.nan
            tenor_days = np.nan
            disbursed_date = pd.NaT
            loan_status = "rejected"
            defaulted = None
            days_late = np.nan
            repaid_pct = np.nan

        records.append({
            "loan_id": loan_id_counter,
            "customer_id": cid,
            "acquisition_channel": channel[i],
            "application_date": app_date,
            "requested_amount_kes": requested_amount,
            "internal_risk_tier_at_application": risk_tier,
            "internal_default_prob_at_application": round(default_prob, 4),
            "approved": int(approved),
            "approved_amount_kes": approved_amount,
            "interest_rate_monthly_pct": None if pd.isna(interest_rate_monthly_pct) else round(float(interest_rate_monthly_pct), 2),
            "tenor_days": tenor_days,
            "disbursed_date": disbursed_date,
            "loan_status": loan_status,
            "defaulted": defaulted,
            "days_late": days_late,
            "repaid_pct_of_principal": repaid_pct,
            "prior_completed_loans_with_us": prior_completed,
            "prior_defaults_with_us": prior_defaults,
            "prior_avg_days_late_with_us": round(prior_avg_days_late, 1),
            "prior_repayment_rate_with_us": round(prior_repayment_rate, 3),
        })

        if approved and loan_status in ("closed_good", "closed_default", "written_off"):
            prior_completed += 1
            if defaulted:
                prior_defaults += 1
            prior_avg_days_late = (prior_avg_days_late * (prior_completed - 1) + days_late) / prior_completed
            prior_repayment_rate = (prior_repayment_rate * (prior_completed - 1) + repaid_pct) / prior_completed

applications = pd.DataFrame(records)
applications.to_csv(f"{OUT_DIR}/applications.csv", index=False)
print(applications.shape)
print(applications["loan_status"].value_counts())
applications.head()

(19438, 20)
loan_status
rejected          10860
closed_good        4330
closed_default     2782
written_off        1466
Name: count, dtype: int64


,loan_id,customer_id,acquisition_channel,application_date,requested_amount_kes,internal_risk_tier_at_application,internal_default_prob_at_application,approved,approved_amount_kes,interest_rate_monthly_pct,tenor_days,disbursed_date,loan_status,defaulted,days_late,repaid_pct_of_principal,prior_completed_loans_with_us,prior_defaults_with_us,prior_avg_days_late_with_us,prior_repayment_rate_with_us
0,500001,100000,sales_assisted,2025-01-12,24100.0,D,0.9000,0,0.0,NaN,NaN,NaT,rejected,None,NaN,NaN,0,0,0.0,1.000
1,500002,100001,digital,2025-05-03,25400.0,D,0.9000,1,12400.0,23.73,30.0,2025-05-03,closed_default,True,190.0,0.271,0,0,0.0,1.000
2,500003,100001,digital,2025-06-20,20600.0,D,0.9000,1,10200.0,23.09,120.0,2025-06-20,closed_default,True,90.0,0.091,1,1,190.0,0.271
3,500004,100002,sales_assisted,2024-10-05,20500.0,D,0.6976,1,9700.0,19.74,120.0,2024-10-05,closed_good,False,6.0,1.000,0,0,0.0,1.000
4,500005,100003,sales_assisted,2023-08-23,83500.0,D,0.6938,0,0.0,NaN,NaN,NaT,rejected,None,NaN,NaN,0,0,0.0,1.000


## PAYMENT HISTORY TABLE (one row per closed/active loan, lender-side ledger)

In [7]:
payment_history = applications.loc[
    applications["approved"] == 1,
    ["loan_id", "customer_id", "disbursed_date", "tenor_days", "loan_status",
     "days_late", "repaid_pct_of_principal", "approved_amount_kes"]
].copy()
payment_history = payment_history.rename(columns={
    "days_late": "days_late_final",
    "repaid_pct_of_principal": "repaid_pct_final"
})
payment_history["due_date"] = payment_history["disbursed_date"] + pd.to_timedelta(
    payment_history["tenor_days"], unit="D"
)

payment_history.to_csv(f"{OUT_DIR}/payment_history.csv", index=False)
print("payment_history:", payment_history.shape)

# ----------------------------------------------------------------------------
# 7. MASTER DATASET (model-ready, one row per application)
# ----------------------------------------------------------------------------
master = (
    applications
    .merge(customers.drop(columns=["acquisition_channel"]), on="customer_id", how="left")
    .merge(crb_data, on="customer_id", how="left")
    .merge(bank_statement_features, on="customer_id", how="left")
    .merge(app_behavior, on="customer_id", how="left")
)

# Only closed loans have a definitive default label; rejected/active loans stay NaN
master["target_default"] = master["defaulted"].map({True: 1, False: 0})

master.to_csv(f"{OUT_DIR}/master_dataset.csv", index=False)
print("master_dataset:", master.shape)

print("\n--- CLASS BALANCE (closed loans only) ---")
closed = master[master["target_default"].notna()]
print(closed["target_default"].value_counts(normalize=True).round(4))
print(f"Total closed loans with definitive outcome: {len(closed):,}")

print("\n--- CHANNEL SPLIT ---")
print(master["acquisition_channel"].value_counts())

print("\n--- DEFAULT RATE BY RISK TIER (closed loans) ---")
print(closed.groupby("internal_risk_tier_at_application")["target_default"].agg(["mean", "count"]))

payment_history: (8578, 9)
master_dataset: (19438, 51)

--- CLASS BALANCE (closed loans only) ---
target_default
0.0    0.5048
1.0    0.4952
Name: proportion, dtype: float64
Total closed loans with definitive outcome: 8,578

--- CHANNEL SPLIT ---
acquisition_channel
digital           12123
sales_assisted     7315
Name: count, dtype: int64

--- DEFAULT RATE BY RISK TIER (closed loans) ---
                                       mean  count
internal_risk_tier_at_application                 
A                                  0.058716   1090
B                                  0.184963   1503
C                                  0.347269   1794
D                                  0.783345   4191


## FEATURE ENGINEERING

In [8]:
# ----------------------------------------------------------------------------
# FEATURE ENGINEERING
# ----------------------------------------------------------------------------
NUMERIC_FEATURES = [
    "age", "monthly_income_kes", "crb_score", "num_active_loans_other_lenders",
    "total_outstanding_debt_kes", "num_defaults_last_24m", "max_days_past_due_24m",
    "crb_listed_negative", "avg_monthly_inflow_kes", "avg_monthly_outflow_kes",
    "inflow_volatility_cv", "avg_closing_balance_kes", "salary_regularity_score",
    "num_bounced_payments_6m", "mobile_money_txn_count_monthly",
    "num_distinct_income_sources", "app_tenure_days", "app_sessions_per_week",
    "contacts_permission_granted", "sms_permission_granted", "num_support_tickets_90d",
    "profile_completeness_pct", "days_since_last_app_open",
    "prior_completed_loans_with_us", "prior_defaults_with_us",
    "prior_avg_days_late_with_us", "prior_repayment_rate_with_us",
    "requested_amount_kes",
]

CATEGORICAL_FEATURES = [
    "acquisition_channel", "region", "employment_type", "gender", "device_type",
]

def engineer_features(df):
    out = df.copy()
    out["debt_to_income_ratio"] = (
        out["total_outstanding_debt_kes"] / out["monthly_income_kes"].replace(0, np.nan)
    ).fillna(0)
    out["requested_to_income_ratio"] = (
        out["requested_amount_kes"] / out["monthly_income_kes"].replace(0, np.nan)
    ).fillna(0)
    out["net_cashflow_kes"] = out["avg_monthly_inflow_kes"] - out["avg_monthly_outflow_kes"]
    out["outflow_to_inflow_ratio"] = (
        out["avg_monthly_outflow_kes"] / out["avg_monthly_inflow_kes"].replace(0, np.nan)
    ).fillna(1.0)
    out["is_digital_channel"] = (out["acquisition_channel"] == "digital").astype(int)

    app_cols = [
        "app_tenure_days", "app_sessions_per_week", "contacts_permission_granted",
        "sms_permission_granted", "num_support_tickets_90d",
        "profile_completeness_pct", "days_since_last_app_open",
    ]
    for c in app_cols:
        out[c] = out[c].fillna(-1)

    out["device_type"] = out["device_type"].fillna("no_app_no_device")
    return out

NUMERIC_FEATURES_ENGINEERED = NUMERIC_FEATURES + [
    "debt_to_income_ratio", "requested_to_income_ratio",
    "net_cashflow_kes", "outflow_to_inflow_ratio", "is_digital_channel",
]
ALL_FEATURES_ENGINEERED = NUMERIC_FEATURES_ENGINEERED + CATEGORICAL_FEATURES

df_feat = engineer_features(master)
print(df_feat.shape)
df_feat[["customer_id","debt_to_income_ratio","requested_to_income_ratio","net_cashflow_kes","is_digital_channel"]].head()

(19438, 56)


,customer_id,debt_to_income_ratio,requested_to_income_ratio,net_cashflow_kes,is_digital_channel
0,100000,1.175510,0.983673,2700.0,0
1,100001,0.808581,0.838284,8600.0,1
2,100001,0.808581,0.679868,8600.0,1
3,100002,0.000000,0.654952,5600.0,0
4,100003,0.392338,1.103038,4500.0,0


## Train the PD Model (LightGBM)

In [9]:
!pip install lightgbm scikit-learn joblib --quiet

In [10]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    classification_report, roc_curve
)
import joblib
import json

os.makedirs("models", exist_ok=True)

# Only closed loans have a definitive label
closed = df_feat[df_feat["target_default"].notna()].copy()
closed["target_default"] = closed["target_default"].astype(int)

print(f"Training on {len(closed):,} closed loans "
      f"({closed['target_default'].mean():.1%} default rate)")

for c in CATEGORICAL_FEATURES:
    closed[c] = closed[c].astype("category")

X = closed[ALL_FEATURES_ENGINEERED]
y = closed["target_default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42, stratify=y_train
)

train_set = lgb.Dataset(X_train, label=y_train, categorical_feature=CATEGORICAL_FEATURES)
val_set = lgb.Dataset(X_val, label=y_val, categorical_feature=CATEGORICAL_FEATURES, reference=train_set)

params = {
    "objective": "binary",
    "metric": ["auc", "binary_logloss"],
    "learning_rate": 0.03,
    "num_leaves": 31,
    "min_data_in_leaf": 40,
    "feature_fraction": 0.85,
    "bagging_fraction": 0.85,
    "bagging_freq": 5,
    "lambda_l2": 2.0,
    "verbosity": -1,
    "seed": 42,
}

model = lgb.train(
    params,
    train_set,
    num_boost_round=2000,
    valid_sets=[train_set, val_set],
    valid_names=["train", "val"],
    callbacks=[lgb.early_stopping(stopping_rounds=75), lgb.log_evaluation(period=0)],
)

print(f"\nBest iteration: {model.best_iteration}")

test_pred = model.predict(X_test, num_iteration=model.best_iteration)
auc = roc_auc_score(y_test, test_pred)
ap = average_precision_score(y_test, test_pred)
brier = brier_score_loss(y_test, test_pred)

print(f"\n=== TEST SET PERFORMANCE ===")
print(f"ROC-AUC: {auc:.4f}")
print(f"PR-AUC:  {ap:.4f}")
print(f"Brier:   {brier:.4f}")

fpr, tpr, thresholds = roc_curve(y_test, test_pred)
ks = max(tpr - fpr)
gini = 2 * auc - 1
print(f"KS statistic: {ks:.4f}")
print(f"Gini coefficient: {gini:.4f}")

print("\nClassification report @ 0.5 threshold:")
print(classification_report(y_test, (test_pred >= 0.5).astype(int)))

importance = pd.DataFrame({
    "feature": model.feature_name(),
    "gain": model.feature_importance(importance_type="gain"),
}).sort_values("gain", ascending=False)
print("\n=== TOP 15 FEATURES BY GAIN ===")
print(importance.head(15).to_string(index=False))

model.save_model("models/pd_model.txt")
joblib.dump(model, "models/pd_model.pkl")
print("\nModel saved to models/pd_model.pkl")

Training on 8,578 closed loans (49.5% default rate)
Training until validation scores don't improve for 75 rounds
Early stopping, best iteration is:
[131]	train's auc: 0.905568	train's binary_logloss: 0.412071	val's auc: 0.813484	val's binary_logloss: 0.530246

Best iteration: 131

=== TEST SET PERFORMANCE ===
ROC-AUC: 0.8249
PR-AUC:  0.8035
Brier:   0.1688
KS statistic: 0.5319
Gini coefficient: 0.6499

Classification report @ 0.5 threshold:
              precision    recall  f1-score   support

           0       0.76      0.78      0.77       866
           1       0.77      0.75      0.76       850

    accuracy                           0.77      1716
   macro avg       0.77      0.77      0.77      1716
weighted avg       0.77      0.77      0.77      1716


=== TOP 15 FEATURES BY GAIN ===
                       feature         gain
                     crb_score 19722.803741
       salary_regularity_score  4958.388324
   prior_avg_days_late_with_us  3407.984025
               empl

## Per-channel performance check

In [11]:
test_df = X_test.copy()
test_df["y_true"] = y_test.values
test_df["y_pred"] = test_pred

print("=== AUC BY CHANNEL (unified model, sliced) ===")
for ch, grp in test_df.groupby("acquisition_channel", observed=True):
    if grp["y_true"].nunique() > 1:
        print(f"  {ch}: AUC={roc_auc_score(grp['y_true'], grp['y_pred']):.4f}  n={len(grp)}")

=== AUC BY CHANNEL (unified model, sliced) ===
  digital: AUC=0.8325  n=1111
  sales_assisted: AUC=0.8111  n=605


## POLICY ENGINE — turns PD score into approve/decline + terms

In [12]:
def assign_risk_tier(pd_score: float) -> str:
    if pd_score < 0.12:
        return "A"
    elif pd_score < 0.25:
        return "B"
    elif pd_score < 0.45:
        return "C"
    else:
        return "D"

# Channel-specific approval policy — tune these against real portfolio data later
APPROVAL_POLICY = {
    "digital":        {"A": True, "B": True, "C": True,  "D": False},
    "sales_assisted": {"A": True, "B": True, "C": True,  "D": False},
}
# Max PD hard-decline threshold, overrides tier logic as a safety net
HARD_DECLINE_PD = 0.55

# Amount approval ratio & pricing by tier
TIER_TERMS = {
    "A": {"amount_ratio": 1.00, "rate_range": (6, 9)},
    "B": {"amount_ratio": 0.90, "rate_range": (9, 13)},
    "C": {"amount_ratio": 0.70, "rate_range": (13, 18)},
    "D": {"amount_ratio": 0.00, "rate_range": (18, 24)},  # declined, kept for reference
}

def score_application(pd_score: float, channel: str, requested_amount: float) -> dict:
    """Given a PD score, channel, and requested amount -> full lending decision."""
    tier = assign_risk_tier(pd_score)
    hard_decline = pd_score >= HARD_DECLINE_PD
    policy_approve = APPROVAL_POLICY.get(channel, APPROVAL_POLICY["digital"]).get(tier, False)
    approved = policy_approve and not hard_decline

    terms = TIER_TERMS[tier]
    if approved:
        approved_amount = round(requested_amount * terms["amount_ratio"], -2)
        rate_lo, rate_hi = terms["rate_range"]
        # price slightly worse within the tier band as PD rises toward the tier's ceiling
        interest_rate = round(rate_lo + (rate_hi - rate_lo) * min(pd_score / 0.45, 1.0), 2)
    else:
        approved_amount = 0.0
        interest_rate = None

    return {
        "pd_score": round(pd_score, 4),
        "risk_tier": tier,
        "approved": approved,
        "approved_amount_kes": approved_amount,
        "interest_rate_monthly_pct": interest_rate,
        "decline_reason": "hard_decline_pd_threshold" if hard_decline else
                           ("policy_tier_decline" if not policy_approve else None),
    }

# --- Apply to the test set to sanity-check the policy layer ---
policy_results = []
for idx, (pd_score, ch, req_amt) in enumerate(zip(
    test_pred, test_df["acquisition_channel"], test_df["requested_amount_kes"]
)):
    r = score_application(pd_score, ch, req_amt)
    r["y_true_default"] = test_df["y_true"].iloc[idx]
    policy_results.append(r)

policy_df = pd.DataFrame(policy_results)

print("=== POLICY OUTCOME DISTRIBUTION ===")
print(policy_df["risk_tier"].value_counts())
print("\n=== APPROVAL RATE BY TIER ===")
print(policy_df.groupby("risk_tier")["approved"].mean())
print("\n=== ACTUAL DEFAULT RATE AMONG APPROVED, BY TIER (policy validation) ===")
print(policy_df[policy_df["approved"]].groupby("risk_tier")["y_true_default"].agg(["mean", "count"]))

overall_approval_rate = policy_df["approved"].mean()
overall_default_rate_approved = policy_df[policy_df["approved"]]["y_true_default"].mean()
print(f"\nOverall approval rate under policy: {overall_approval_rate:.1%}")
print(f"Default rate among approved loans: {overall_default_rate_approved:.1%}")

=== POLICY OUTCOME DISTRIBUTION ===
risk_tier
D    886
B    406
C    304
A    120
Name: count, dtype: int64

=== APPROVAL RATE BY TIER ===
risk_tier
A    1.0
B    1.0
C    1.0
D    0.0
Name: approved, dtype: float64

=== ACTUAL DEFAULT RATE AMONG APPROVED, BY TIER (policy validation) ===
               mean  count
risk_tier                 
A          0.075000    120
B          0.184729    406
C          0.345395    304

Overall approval rate under policy: 48.4%
Default rate among approved loans: 22.8%


In [13]:
# Save the policy engine + feature engineering as .py files for reuse
os.makedirs("src", exist_ok=True)

with open("src/policy_engine.py", "w") as f:
    f.write('''
"""Stage 2: Policy Engine — PD score -> lending decision"""

def assign_risk_tier(pd_score: float) -> str:
    if pd_score < 0.12:
        return "A"
    elif pd_score < 0.25:
        return "B"
    elif pd_score < 0.45:
        return "C"
    else:
        return "D"

APPROVAL_POLICY = {
    "digital":        {"A": True, "B": True, "C": True,  "D": False},
    "sales_assisted": {"A": True, "B": True, "C": True,  "D": False},
}
HARD_DECLINE_PD = 0.55

TIER_TERMS = {
    "A": {"amount_ratio": 1.00, "rate_range": (6, 9)},
    "B": {"amount_ratio": 0.90, "rate_range": (9, 13)},
    "C": {"amount_ratio": 0.70, "rate_range": (13, 18)},
    "D": {"amount_ratio": 0.00, "rate_range": (18, 24)},
}

def score_application(pd_score: float, channel: str, requested_amount: float) -> dict:
    tier = assign_risk_tier(pd_score)
    hard_decline = pd_score >= HARD_DECLINE_PD
    policy_approve = APPROVAL_POLICY.get(channel, APPROVAL_POLICY["digital"]).get(tier, False)
    approved = policy_approve and not hard_decline

    terms = TIER_TERMS[tier]
    if approved:
        approved_amount = round(requested_amount * terms["amount_ratio"], -2)
        rate_lo, rate_hi = terms["rate_range"]
        interest_rate = round(rate_lo + (rate_hi - rate_lo) * min(pd_score / 0.45, 1.0), 2)
    else:
        approved_amount = 0.0
        interest_rate = None

    return {
        "pd_score": round(pd_score, 4),
        "risk_tier": tier,
        "approved": approved,
        "approved_amount_kes": approved_amount,
        "interest_rate_monthly_pct": interest_rate,
        "decline_reason": "hard_decline_pd_threshold" if hard_decline else
                           ("policy_tier_decline" if not policy_approve else None),
    }
''')

with open("src/features.py", "w") as f:
    f.write(f'''
"""Feature engineering — shared between training and serving"""
import numpy as np
import pandas as pd

NUMERIC_FEATURES = {NUMERIC_FEATURES!r}
CATEGORICAL_FEATURES = {CATEGORICAL_FEATURES!r}

def engineer_features(df):
    out = df.copy()
    out["debt_to_income_ratio"] = (
        out["total_outstanding_debt_kes"] / out["monthly_income_kes"].replace(0, np.nan)
    ).fillna(0)
    out["requested_to_income_ratio"] = (
        out["requested_amount_kes"] / out["monthly_income_kes"].replace(0, np.nan)
    ).fillna(0)
    out["net_cashflow_kes"] = out["avg_monthly_inflow_kes"] - out["avg_monthly_outflow_kes"]
    out["outflow_to_inflow_ratio"] = (
        out["avg_monthly_outflow_kes"] / out["avg_monthly_inflow_kes"].replace(0, np.nan)
    ).fillna(1.0)
    out["is_digital_channel"] = (out["acquisition_channel"] == "digital").astype(int)

    app_cols = [
        "app_tenure_days", "app_sessions_per_week", "contacts_permission_granted",
        "sms_permission_granted", "num_support_tickets_90d",
        "profile_completeness_pct", "days_since_last_app_open",
    ]
    for c in app_cols:
        out[c] = out[c].fillna(-1)
    out["device_type"] = out["device_type"].fillna("no_app_no_device")
    return out

NUMERIC_FEATURES_ENGINEERED = NUMERIC_FEATURES + [
    "debt_to_income_ratio", "requested_to_income_ratio",
    "net_cashflow_kes", "outflow_to_inflow_ratio", "is_digital_channel",
]
ALL_FEATURES_ENGINEERED = NUMERIC_FEATURES_ENGINEERED + CATEGORICAL_FEATURES
''')

print("Saved src/policy_engine.py and src/features.py")
print(os.listdir("src"))

Saved src/policy_engine.py and src/features.py
['features.py', 'policy_engine.py']


##  Save model metadata

In [14]:
import json

metadata = {
    "model_type": "LightGBM binary classifier",
    "trained_on_rows": int(len(closed)),
    "test_auc": round(float(auc), 4),
    "test_gini": round(float(gini), 4),
    "test_ks": round(float(ks), 4),
    "features_numeric": NUMERIC_FEATURES_ENGINEERED,
    "features_categorical": CATEGORICAL_FEATURES,
    "categorical_levels": {
        c: sorted(closed[c].astype(str).unique().tolist()) for c in CATEGORICAL_FEATURES
    },
}
with open("models/model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved models/model_metadata.json")
print(json.dumps(metadata, indent=2)[:800])

Saved models/model_metadata.json
{
  "model_type": "LightGBM binary classifier",
  "trained_on_rows": 8578,
  "test_auc": 0.8249,
  "test_gini": 0.6499,
  "test_ks": 0.5319,
  "features_numeric": [
    "age",
    "monthly_income_kes",
    "crb_score",
    "num_active_loans_other_lenders",
    "total_outstanding_debt_kes",
    "num_defaults_last_24m",
    "max_days_past_due_24m",
    "crb_listed_negative",
    "avg_monthly_inflow_kes",
    "avg_monthly_outflow_kes",
    "inflow_volatility_cv",
    "avg_closing_balance_kes",
    "salary_regularity_score",
    "num_bounced_payments_6m",
    "mobile_money_txn_count_monthly",
    "num_distinct_income_sources",
    "app_tenure_days",
    "app_sessions_per_week",
    "contacts_permission_granted",
    "sms_permission_granted",
    "num_support_tickets_90d",
    "profile_complete


## Writing the FastAPI app to disk

In [18]:
features_code = '''
"""Feature engineering - shared between training and serving"""
import numpy as np
import pandas as pd

NUMERIC_FEATURES = ''' + repr(NUMERIC_FEATURES) + '''
CATEGORICAL_FEATURES = ''' + repr(CATEGORICAL_FEATURES) + '''

def engineer_features(df):
    out = df.copy()
    out["debt_to_income_ratio"] = (
        out["total_outstanding_debt_kes"] / out["monthly_income_kes"].replace(0, np.nan)
    ).fillna(0)
    out["requested_to_income_ratio"] = (
        out["requested_amount_kes"] / out["monthly_income_kes"].replace(0, np.nan)
    ).fillna(0)
    out["net_cashflow_kes"] = out["avg_monthly_inflow_kes"] - out["avg_monthly_outflow_kes"]
    out["outflow_to_inflow_ratio"] = (
        out["avg_monthly_outflow_kes"] / out["avg_monthly_inflow_kes"].replace(0, np.nan)
    ).fillna(1.0)
    out["is_digital_channel"] = (out["acquisition_channel"] == "digital").astype(int)

    app_cols = [
        "app_tenure_days", "app_sessions_per_week", "contacts_permission_granted",
        "sms_permission_granted", "num_support_tickets_90d",
        "profile_completeness_pct", "days_since_last_app_open",
    ]
    for c in app_cols:
        out[c] = out[c].fillna(-1)
    out["device_type"] = out["device_type"].fillna("no_app_no_device")
    return out

NUMERIC_FEATURES_ENGINEERED = NUMERIC_FEATURES + [
    "debt_to_income_ratio", "requested_to_income_ratio",
    "net_cashflow_kes", "outflow_to_inflow_ratio", "is_digital_channel",
]
ALL_FEATURES_ENGINEERED = NUMERIC_FEATURES_ENGINEERED + CATEGORICAL_FEATURES
'''

with open("src/features.py", "w", encoding="utf-8") as f:
    f.write(features_code)

print("Rewritten src/features.py with UTF-8 encoding")

# Also rewrite policy_engine.py with explicit UTF-8 just to be safe
policy_code = '''
"""Stage 2: Policy Engine - PD score to lending decision"""

def assign_risk_tier(pd_score):
    if pd_score < 0.12:
        return "A"
    elif pd_score < 0.25:
        return "B"
    elif pd_score < 0.45:
        return "C"
    else:
        return "D"

APPROVAL_POLICY = {
    "digital":        {"A": True, "B": True, "C": True,  "D": False},
    "sales_assisted": {"A": True, "B": True, "C": True,  "D": False},
}
HARD_DECLINE_PD = 0.55

TIER_TERMS = {
    "A": {"amount_ratio": 1.00, "rate_range": (6, 9)},
    "B": {"amount_ratio": 0.90, "rate_range": (9, 13)},
    "C": {"amount_ratio": 0.70, "rate_range": (13, 18)},
    "D": {"amount_ratio": 0.00, "rate_range": (18, 24)},
}

def score_application(pd_score, channel, requested_amount):
    tier = assign_risk_tier(pd_score)
    hard_decline = pd_score >= HARD_DECLINE_PD
    policy_approve = APPROVAL_POLICY.get(channel, APPROVAL_POLICY["digital"]).get(tier, False)
    approved = policy_approve and not hard_decline

    terms = TIER_TERMS[tier]
    if approved:
        approved_amount = round(requested_amount * terms["amount_ratio"], -2)
        rate_lo, rate_hi = terms["rate_range"]
        interest_rate = round(rate_lo + (rate_hi - rate_lo) * min(pd_score / 0.45, 1.0), 2)
    else:
        approved_amount = 0.0
        interest_rate = None

    return {
        "pd_score": round(pd_score, 4),
        "risk_tier": tier,
        "approved": approved,
        "approved_amount_kes": approved_amount,
        "interest_rate_monthly_pct": interest_rate,
        "decline_reason": "hard_decline_pd_threshold" if hard_decline else
                           ("policy_tier_decline" if not policy_approve else None),
    }
'''

with open("src/policy_engine.py", "w", encoding="utf-8") as f:
    f.write(policy_code)

print("Rewritten src/policy_engine.py with UTF-8 encoding")

Rewritten src/features.py with UTF-8 encoding
Rewritten src/policy_engine.py with UTF-8 encoding
